# 01 · Data audit

Verify the dataset, the patient-ID parsing, and the patient-level structure that justifies leakage-resistant splitting.

In [1]:
import os
from pathlib import Path
# Run from the repo root so the package's relative paths (data/, outputs/) resolve.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('cwd:', Path.cwd())

cwd: /home/emre/Projeler/MalariaCellsThesis


## File counts

In [2]:
from malaria_vit.config import RAW_DIR
from malaria_vit.data import build_image_index
idx = build_image_index(RAW_DIR)
print('total images:', len(idx))
print(idx['label'].value_counts().rename({0: 'uninfected', 1: 'parasitised'}))

total images: 27558
label
parasitised    13779
uninfected     13779
Name: count, dtype: int64


## Patient structure

In [3]:
par = set(idx.loc[idx.label == 1, 'patient_id'])
uni = set(idx.loc[idx.label == 0, 'patient_id'])
print('distinct patients:', idx['patient_id'].nunique())
print('parasitised patients:', len(par))
print('uninfected patients:', len(uni))
print('patients in BOTH classes:', len(par & uni))

distinct patients: 200
parasitised patients: 150
uninfected patients: 200
patients in BOTH classes: 150


**Finding:** 200 patients (150 infected + 50 healthy); 150 appear in *both* classes, so a random image-level split would place the same patient in train and test. This is the concrete justification for patient-level splitting.

## The split — leakage check

In [4]:
import pandas as pd
from malaria_vit.config import SPLIT_CSV
df = pd.read_csv(SPLIT_CSV)
print('patients spanning >1 split (must be 0):',
      int((df.groupby('patient_id')['split'].nunique() > 1).sum()))
print('\npatients per split:'); print(df.drop_duplicates('patient_id')['split'].value_counts())
print('\ncells per split:'); print(df['split'].value_counts())
print('\nclass balance per split:'); print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

patients spanning >1 split (must be 0): 0

patients per split:
split
train    140
test      30
val       30
Name: count, dtype: int64

cells per split:
split
train    19897
test      4074
val       3587
Name: count, dtype: int64

class balance per split:
label     0      1
split             
test   2049   2025
train  9665  10232
val    2065   1522
